In [1]:
# =====================================================================
# SECTION 0 — ⚙️ CONFIGURATION
# Change only these paths/variables for any new dataset
# =====================================================================
import pandas as pd
import numpy as np
import joblib
import re
import warnings
warnings.filterwarnings('ignore')

PICK_DATA_PATH    = '/home/adityaa/Desktop/internship/data/tbl_pick_summary.csv'
SPACE_DATA_PATH   = '/home/adityaa/Desktop/internship/data/tblspace.csv'
PUTAWAY_DATA_PATH = '/home/adityaa/Desktop/internship/data/putaway_summary.csv'
MODEL_PATH        = 'hybrid_production_model.pkl'
SCORE_YEAR        = 2026  # Year to score — change for any period
OUTPUT_CSV        = 'Item_Zone_Efficiency_Predictions_2026.csv'

print(f"--- Pipeline Initialized for Year: {SCORE_YEAR} ---")

# =====================================================================
# SECTION 1 — 📦 LOAD FROZEN MODEL ARTIFACTS
# =====================================================================
try:
    model_payload = joblib.load(MODEL_PATH)
    print("✅ Model artifacts loaded successfully.")
except FileNotFoundError:
    raise FileNotFoundError(f"❌ CRITICAL ERROR: {MODEL_PATH} not found. Run the training notebook first.")

# Extract all keys for easy access later
scaler = model_payload['scaler']
tier_mapping = model_payload['tier_mapping']
throughput_95th_ceiling_global = model_payload['throughput_95th_ceiling_global']
putaway_75th_ceiling_global = model_payload['putaway_75th_ceiling_global']
warehouse_median_rate = model_payload['warehouse_median_rate']
d2s_baseline_seconds = model_payload['d2s_baseline_seconds']
drift_baseline = model_payload.get('drift_baseline', 0.0) 

# 🚨 INFERENCE KEY: Use the updated item-level parameter list
features_to_process = model_payload['features_to_cluster'] 

# Extract the dictionaries containing all 3 tracks
models_dict = model_payload['models']   # Contains 'data', 'strategy', 'hybrid'
weights_dict = model_payload['weights'] # Contains 'data', 'strategy', 'hybrid'

print("✅ Extracted all baselines and frozen parameters.")

# =====================================================================
# SECTION 2 — 📥 LOAD AND PREPROCESS RAW DATA
# =====================================================================
table1_cols = ["summary_id" ,"pickDate", "count", "itemCode", "itemDescription", "pickQty", "ordQty", "pickedStorageBin", "pickerId", "pickHeNo", "storageSectionId", "totalTimeSeconds", "perQtyTime"]
# 🚨 INFERENCE KEY: Ensure itemCode is pulled from Space data to support Broadcast logic in Section 3
table2_cols = ["space_id", "st_sec_id", "totalCount", "usedCount", "createdDate", "utilization", "itemCode"]

# Load Raw Data
table1 = pd.read_csv(PICK_DATA_PATH, header=None, names=table1_cols)
table2 = pd.read_csv(SPACE_DATA_PATH, header=None, names=table2_cols)
table3 = pd.read_csv(PUTAWAY_DATA_PATH, header=1, low_memory=False)
table3.columns = ['summary_id', 'putawayDate', 'count', 'itemCode', 'putAwayQty',
                     'ordQty', 'storageBin', 'storageSectionId', 'totalTimeinSeconds',
                     'perQtyTime', 'item_text', 'proposed_st_bin']

# Parse Dates
table1['pickDate'] = pd.to_datetime(table1['pickDate'], errors='coerce')
table2['createdDate'] = pd.to_datetime(table2['createdDate'], errors='coerce')
table3['putaway_Date'] = pd.to_datetime(table3['putawayDate'], format='%Y-%m-%d', errors='coerce')

# Filter strictly to the SCORE_YEAR
table1 = table1[table1['pickDate'].dt.year == SCORE_YEAR].copy()
table2 = table2[table2['createdDate'].dt.year == SCORE_YEAR].copy()
table3 = table3[table3['putaway_Date'].dt.year == SCORE_YEAR].copy()

# Zone Validation Function
def is_valid_warehouse_zone(zone_str):
    if pd.isna(zone_str):
        return False
    zone_str = str(zone_str).strip().upper()
    # List of allowed canonical zones
    named_exceptions = ['AA', 'AB', 'AC', 'AD', 'AE', 'AF', 'AG', 'AH', 'AI', 'AJ', 'AK', 'AL', 'BA', 'BB', 'BC', 'BD', 'BE', 'BF', 'BG', 'BH', 'BI', 'BJ', 'DP', 'LOC']  
    if zone_str in named_exceptions:
        return True
    return bool(re.match(r'^[A-Z]{2,3}$', zone_str))

# Apply Zone Filters
table1 = table1[table1['storageSectionId'].apply(is_valid_warehouse_zone)].copy()
table2 = table2[table2['st_sec_id'].apply(is_valid_warehouse_zone)].copy()
table3 = table3[table3['storageSectionId'].apply(is_valid_warehouse_zone)].copy()

# Floor filter on pick time (30s) mandated by Training rules
table1 = table1[(table1['totalTimeSeconds'] >= 30) & (table1['totalTimeSeconds'] <= 7200)]
table1 = table1[(table1['pickQty'] >= 0) & (table1['ordQty'] > 0)]

table2 = table2[table2['totalCount'] > 0]

table3 = table3[(table3['totalTimeinSeconds'] > 0) & (table3['totalTimeinSeconds'] <= 3600)]
table3 = table3[table3['putAwayQty'] > 0]

print(f"✅ Raw Data Preprocessing Complete for {SCORE_YEAR}.")
print(f"   -> Table 1 (Picks) Rows  : {len(table1)}")
print(f"   -> Table 2 (Space) Rows  : {len(table2)}")
print(f"   -> Table 3 (Putaway) Rows: {len(table3)}")

--- Pipeline Initialized for Year: 2026 ---
✅ Model artifacts loaded successfully.
✅ Extracted all baselines and frozen parameters.
✅ Raw Data Preprocessing Complete for 2026.
   -> Table 1 (Picks) Rows  : 70592
   -> Table 2 (Space) Rows  : 338329
   -> Table 3 (Putaway) Rows: 6124


In [2]:
# =====================================================================
# SECTION 3 — ⚙️ ENGINEER ALL 8 PARAMETERS INDEPENDENTLY (ITEM LEVEL)
# Using Frozen 2025 Baselines where applicable to prevent data leakage
# =====================================================================
import pandas as pd
import numpy as np

# Format dates to consistent strings for grouping (Now called 'month' to match training)
table1['month'] = pd.to_datetime(table1['pickDate']).dt.to_period('M').astype(str)
table2['month'] = pd.to_datetime(table2['createdDate']).dt.to_period('M').astype(str)
table3['month'] = pd.to_datetime(table3['putawayDate']).dt.to_period('M').astype(str)

# --- Parameter 1: Item Pick Rate Score ---
pr_df = table1.groupby(['itemCode', 'month', 'storageSectionId']).agg(
    item_picks=('pickQty', 'sum'), item_time=('totalTimeSeconds', 'sum')
).reset_index()

# Convert seconds to hours
pr_df['item_pick_rate'] = pr_df['item_picks'] / ((pr_df['item_time'] / 3600) + 0.001)

# 🚨 FIX: Use frozen per-item target rates from training
sku_median_rates = model_payload['sku_median_rates']
pr_df = pr_df.merge(sku_median_rates, on='itemCode', how='left')
pr_df['target_rate'] = pr_df['target_rate'].fillna(warehouse_median_rate)
pr_df['Item_Pick_Rate_Score'] = ((pr_df['item_pick_rate'] / (pr_df['target_rate'] + 0.0001)) * 100).clip(upper=100.0)

# --- Parameter 2: Item Throughput Score ---
# 🚨 INFERENCE KEY: Using model['throughput_95th_ceiling_global'] instead of recomputing
# Correct — compute throughput in units/second to match training
pr_df['item_throughput'] = pr_df['item_picks'] / (pr_df['item_time'] + 0.001)  # units/sec
zt_df = pr_df.copy()
zt_df['Item_Throughput_Score'] = ((zt_df['item_throughput'] / throughput_95th_ceiling_global) * 100).clip(upper=100.0)

# --- Parameter 3: Item Picker Performance (Volume-Weighted Collapse) ---
# 1. Score the individual picker for every row using 2025 baseline
table1['picker_rate_raw'] = table1['pickQty'] / ((table1['totalTimeSeconds'] / 3600) + 0.001)
table1['picker_prod_score'] = (table1['picker_rate_raw'] / warehouse_median_rate) * 100
table1['picker_prod_score'] = table1['picker_prod_score'].clip(upper=100.0)

# 2. Weight the score by the volume of units handled in that specific transaction
table1['weighted_picker'] = table1['picker_prod_score'] * table1['pickQty']

# 3. Collapse down to Item-Zone-Month grain
zone_picker_df = table1.groupby(['itemCode', 'month', 'storageSectionId']).agg(
    sum_weighted_picker=('weighted_picker', 'sum'),
    sum_pick_qty=('pickQty', 'sum')
).reset_index()
zone_picker_df['Item_Picker_Performance_Score'] = zone_picker_df['sum_weighted_picker'] / (zone_picker_df['sum_pick_qty'] + 0.001)

# --- Parameter 4: Space Utilization (Broadcast Logic) ---
# Calculate at Zone level first
space_df_raw = table2.dropna(subset=['st_sec_id', 'itemCode']).copy()
space_df_raw.rename(columns={'st_sec_id': 'storageSectionId'}, inplace=True)

zm_space = space_df_raw.groupby(['storageSectionId', 'month'])['utilization'].mean().reset_index()
zm_space.rename(columns={'utilization': 'Item_Space_Score'}, inplace=True)
zm_space['Item_Space_Score'] = zm_space['Item_Space_Score'].clip(upper=100.0)

# Broadcast back to all unique items that existed in that zone-month
space_df = space_df_raw[['itemCode', 'month', 'storageSectionId']].drop_duplicates()
space_df = space_df.merge(zm_space[['storageSectionId', 'month', 'Item_Space_Score']], on=['storageSectionId', 'month'], how='left')

# --- Parameter 5: Item Equipment Score (Volume-Weighted Collapse) ---
# 1. Baseline the equipment
equip_stats = table1.groupby(['pickHeNo', 'month', 'storageSectionId']).agg(
    active_equip_hours=('totalTimeSeconds', lambda x: x.sum() / 3600),
    total_machine_picks=('pickQty', 'sum')
).reset_index()

# Compute picks per hour BEFORE merging baselines
equip_stats['Picks_per_Equipment_Hour'] = equip_stats['total_machine_picks'] / (equip_stats['active_equip_hours'] + 0.0001)

machine_baselines = model_payload['machine_baselines']
equip_stats = equip_stats.merge(machine_baselines, on='pickHeNo', how='left')

# Now Picks_per_Equipment_Hour exists, safe to use as fallback
equip_stats['Specific_Machine_Baseline'] = equip_stats['Specific_Machine_Baseline'].fillna(
    equip_stats['Picks_per_Equipment_Hour'].median()
)
equip_stats['Rel_Equip_Eff'] = ((equip_stats['Picks_per_Equipment_Hour'] / (equip_stats['Specific_Machine_Baseline'] + 0.0001)) * 100).clip(upper=100.0)

# 2. Map machine efficiency back to transactions and weight by volume
t1_equip = table1[['itemCode', 'month', 'storageSectionId', 'pickHeNo', 'pickQty']].merge(
    equip_stats[['pickHeNo', 'month', 'storageSectionId', 'Rel_Equip_Eff']], on=['pickHeNo', 'month', 'storageSectionId'], how='left'
)
t1_equip['weighted_equip'] = t1_equip['Rel_Equip_Eff'] * t1_equip['pickQty']

# 3. Collapse down to Item-Zone-Month grain
zone_equip_df = t1_equip.groupby(['itemCode', 'month', 'storageSectionId'])[['weighted_equip', 'pickQty']].sum().reset_index()
zone_equip_df['Item_Equipment_Score'] = zone_equip_df['weighted_equip'] / (zone_equip_df['pickQty'] + 0.001)

# --- Parameter 6: Putaway Throughput Score ---
pt_df = table3.groupby(['itemCode', 'month', 'storageSectionId']).agg(
    qty=('putAwayQty', 'sum'), time=('totalTimeinSeconds', 'sum')
).reset_index()
pt_df['putaway_rate'] = pt_df['qty'] / (pt_df['time'] + 0.001)  # units per second — matches training
pt_df['Item_Putaway_Throughput_Score'] = ((pt_df['putaway_rate'] / putaway_75th_ceiling_global) * 100).clip(upper=100.0)

# --- Parameter 7: Dock-to-Stock (Absolute Conversion instead of Rank) ---
d2s_df = table3.copy()
d2s_df['time_qty_product'] = d2s_df['totalTimeinSeconds'] * d2s_df['putAwayQty']
d2s_agg = d2s_df.groupby(['itemCode', 'month', 'storageSectionId']).agg(
    sum_time_qty_product=('time_qty_product', 'sum'),
    total_putaway_volume=('putAwayQty', 'sum')
).reset_index()
d2s_agg['weighted_d2s_seconds'] = d2s_agg['sum_time_qty_product'] / (d2s_agg['total_putaway_volume'] + 0.001)

# 🚨 INFERENCE KEY: Using absolute comparison against frozen 2025 d2s baseline
d2s_agg['Item_D2S_Score'] = ((d2s_baseline_seconds / (d2s_agg['weighted_d2s_seconds'] + 0.001)) * 100).clip(upper=100.0)

# --- Parameter 8: Putaway Location Accuracy ---
table3['bin_match'] = (table3['storageBin'] == table3['proposed_st_bin']).astype(int)
acc_df = table3.groupby(['itemCode', 'month', 'storageSectionId']).agg(
    matches=('bin_match', 'sum'), total=('bin_match', 'count')
).reset_index()
acc_df['Item_Putaway_Accuracy_Score'] = (acc_df['matches'] / acc_df['total']) * 100

# =====================================================================
# SECTION 4 — 🧩 ASSEMBLE FEATURE MATRIX & IMPUTE MISSING VALUES
# =====================================================================
KEYS = ['itemCode', 'month', 'storageSectionId']
p_dfs = [pr_df, zt_df, zone_picker_df, space_df, zone_equip_df, pt_df, d2s_agg, acc_df]

# 1. Create the definitive union of all keys across all 8 parameters
df_new = pd.concat([df[KEYS] for df in p_dfs]).drop_duplicates().reset_index(drop=True)

# 2. Chain left-merges
df_new = pd.merge(df_new, pr_df[KEYS + ['Item_Pick_Rate_Score']], on=KEYS, how='left')
df_new = pd.merge(df_new, zt_df[KEYS + ['Item_Throughput_Score']], on=KEYS, how='left')
df_new = pd.merge(df_new, zone_picker_df[KEYS + ['Item_Picker_Performance_Score']], on=KEYS, how='left')
df_new = pd.merge(df_new, space_df[KEYS + ['Item_Space_Score']], on=KEYS, how='left')
df_new = pd.merge(df_new, zone_equip_df[KEYS + ['Item_Equipment_Score']], on=KEYS, how='left')
df_new = pd.merge(df_new, pt_df[KEYS + ['Item_Putaway_Throughput_Score']], on=KEYS, how='left')
df_new = pd.merge(df_new, d2s_agg[KEYS + ['Item_D2S_Score']], on=KEYS, how='left')
df_new = pd.merge(df_new, acc_df[KEYS + ['Item_Putaway_Accuracy_Score']], on=KEYS, how='left')

# 3. Two-Step Vectorised Imputation
# 🚨 INFERENCE KEY: Using frozen features_to_process from model payload
for col in features_to_process:
    # Step 1: Item's own zone median
    item_zone_medians = df_new.groupby(['itemCode', 'storageSectionId'])[col].transform('median')
    df_new[col] = df_new[col].fillna(item_zone_medians)
    
    # Step 2: Global Batch Fallback
    global_median = df_new[col].median()
    df_new[col] = df_new[col].fillna(global_median)
    
    # Absolute value bounds
    df_new[col] = df_new[col].clip(lower=0.0, upper=100.0)

print("✅ Item-Level Feature Matrix Assembled & Missing Values Imputed.")

✅ Item-Level Feature Matrix Assembled & Missing Values Imputed.


In [3]:
# =====================================================================
# SECTION 5 & 6 — 🎯 MULTI-TRACK SCALING, WEIGHTING & SCORING
# =====================================================================
from sklearn.metrics import silhouette_score

print("⚙️ Applying Multi-Track Clustering & Scoring at Item Level...")

# 1. Order columns exactly as the model expects
X_raw = df_new[features_to_process].values

# 2. Apply Frozen 2025 Scaler (DO NOT FIT)
X_scaled = scaler.transform(X_raw)

# Map the PKL dictionary keys to the exact requested CSV output names
tracks = [
    ('data', 'CRITIC'),
    ('strategy', 'AHP'),
    ('hybrid', 'Hybrid')
]

# Variables to hold Hybrid metrics for Section 7 Diagnostics
hybrid_X_weighted = None
hybrid_labels = None
hybrid_distances = None
hybrid_tiers_text = None

# Loop through all 3 algorithms
for dict_key, output_name in tracks:
    current_weights = np.array(weights_dict[dict_key])
    current_kmeans = models_dict[dict_key]
    
    # 🚨 DYNAMIC TIER RESOLUTION 🚨
    center_means = current_kmeans.cluster_centers_.mean(axis=1)
    sorted_idx = np.argsort(center_means) # Sorts from lowest to highest
    
    t3_idx = sorted_idx[0] # Lowest performance
    t2_idx = sorted_idx[1] # Middle performance
    t1_idx = sorted_idx[2] # Highest performance

    cluster_names = {
        t1_idx: 'Tier 1: High Efficiency',
        t2_idx: 'Tier 2: Moderate Efficiency',
        t3_idx: 'Tier 3: Low Efficiency'
    }
    
    # 3. Apply Specific Track Weights
    X_weighted = X_scaled * np.sqrt(current_weights)
    
    # 4. Predict Clusters
    labels = current_kmeans.predict(X_weighted)
    distances = current_kmeans.transform(X_weighted)
    
    # Save Hybrid stats for the diagnostic printout below
    if dict_key == 'hybrid':
        hybrid_X_weighted = X_weighted
        hybrid_labels = labels
        hybrid_distances = distances
        # Map labels to text strings strictly for the Section 7 diagnostic printout
        hybrid_tiers_text = np.vectorize(cluster_names.get)(labels)
    
    # 5. VECTORISED PIECEWISE LOGIC & GUARDRAILS (Replaces slow iterrows)
    d1 = distances[:, t1_idx]
    d2 = distances[:, t2_idx]
    d3 = distances[:, t3_idx]
    
    scores = np.zeros(len(df_new))
    
    c_t3 = (labels == t3_idx)
    c_t2_lower = (labels == t2_idx) & (d3 < d1)
    c_t2_upper = (labels == t2_idx) & (d3 >= d1)
    c_t1 = (labels == t1_idx)

    # Geometric piecewise logic
    scores[c_t3] = 33.33 * (1 - (d3[c_t3] / (d3[c_t3] + d2[c_t3] + 1e-5)))
    scores[c_t2_lower] = 33.33 + (16.67 * (d3[c_t2_lower] / (d2[c_t2_lower] + d3[c_t2_lower] + 1e-5)))
    scores[c_t2_upper] = 50.00 + (16.66 * (1 - (d2[c_t2_upper] / (d1[c_t2_upper] + d2[c_t2_upper] + 1e-5))))
    scores[c_t1] = 66.66 + (33.34 * (d2[c_t1] / (d1[c_t1] + d2[c_t1] + 1e-5)))

    # Absolute Guardrails (Mapped to new Item-Level parameter names)
    pk_rate = df_new['Item_Pick_Rate_Score'].values
    ob_thru = df_new['Item_Throughput_Score'].values
    d2s = df_new['Item_D2S_Score'].values
    pa_thru = df_new['Item_Putaway_Throughput_Score'].values

    zombie_mask = (pk_rate <= 5.0) & (ob_thru <= 5.0) & (d2s <= 10.0) & (pa_thru <= 10.0)
    scores[zombie_mask] = 0.0

    perfect_mask = (pk_rate >= 82.0) & (ob_thru >= 89.0) & (d2s >= 85.0) & (pa_thru >= 85.0)
    scores[perfect_mask] = 100.0

    # Assign dynamically to DataFrame
    df_new[f'{output_name}_Efficiency_Score'] = scores

print("✅ Multi-Track Scoring Logic Applied Successfully.")

# =====================================================================
# SECTION 7 — 🔍 HYBRID ACCURACY, DRIFT & QUALITY METRICS
# =====================================================================
print("\n" + "="*55)
print(" 📈 LIVE INFERENCE DIAGNOSTICS (HYBRID TRACK)")
print("="*55)

# Metric 1: Silhouette (Subsampled to 10k to prevent notebook freezing on 300k+ rows)
if len(hybrid_X_weighted) > 10000:
    sample_idx = np.random.choice(len(hybrid_X_weighted), 10000, replace=False)
    sil = silhouette_score(hybrid_X_weighted[sample_idx], hybrid_labels[sample_idx])
else:
    sil = silhouette_score(hybrid_X_weighted, hybrid_labels)

print(f"Silhouette Score (10k Subsample): {sil:.4f}")
print("Interpretation: >0.5 good fit | 0.3-0.5 acceptable | <0.3 model drift")

# Metric 2: Data Drift Detection
current_drift = np.mean(np.min(hybrid_distances, axis=1))
try:
    drift_threshold = drift_baseline * 1.5
    print(f"\nDrift Score: {current_drift:.4f} | Baseline: {drift_baseline:.4f} | Threshold: {drift_threshold:.4f}")
    if current_drift > drift_threshold and drift_baseline > 0:
        print("⚠️ WARNING: Significant drift detected — model retraining recommended.")
    else:
        print("✅ STABLE: New data fits securely within historical cluster boundaries.")
except Exception as e:
    print(f"\nDrift Score: {current_drift:.4f} | (Baseline skipped)")

# Metric 3: Tier Distribution
tier_counts = pd.Series(hybrid_tiers_text).value_counts()
tier_pct = (tier_counts / len(hybrid_tiers_text) * 100).round(1)
print("\nTier Distribution (New Data):")
for tier, pct in tier_pct.items():
    count = tier_counts[tier]
    print(f" - {tier}: {count} items ({pct}%)")

# =====================================================================
# SECTION 8 — 💾 EXPORT FINAL PREDICTIONS
# =====================================================================
# Only exporting the exact columns requested by your Head of Data Science
output_columns = [
    'month', 'storageSectionId', 'itemCode',
    'CRITIC_Efficiency_Score', 'AHP_Efficiency_Score', 'Hybrid_Efficiency_Score'
]

final_export_df = df_new[output_columns].copy()

# Round all numeric columns to 2 decimal places for clean reporting
numeric_cols = final_export_df.select_dtypes(include=[np.number]).columns
final_export_df[numeric_cols] = final_export_df[numeric_cols].round(2)

# Ensure chronologically sorted and saved
final_export_df = final_export_df.sort_values(by=['month', 'storageSectionId', 'itemCode']).reset_index(drop=True)
final_export_df.to_csv(OUTPUT_CSV, index=False)

print("\n" + "="*55)
print(f" 🎉 SUCCESS: {SCORE_YEAR} Inference Complete!")
print(f" -> Output saved to '{OUTPUT_CSV}'")
print(f" -> Total Records Scored: {len(final_export_df)}")
print("="*55)

print("\n--- Final Predictions Preview ---")
print(final_export_df.head(20).to_string(index=False))

⚙️ Applying Multi-Track Clustering & Scoring at Item Level...
✅ Multi-Track Scoring Logic Applied Successfully.

 📈 LIVE INFERENCE DIAGNOSTICS (HYBRID TRACK)
Silhouette Score (10k Subsample): 0.5009
Interpretation: >0.5 good fit | 0.3-0.5 acceptable | <0.3 model drift

Drift Score: 0.1440 | Baseline: 0.1541 | Threshold: 0.2312
✅ STABLE: New data fits securely within historical cluster boundaries.

Tier Distribution (New Data):
 - Tier 1: High Efficiency: 44888 items (70.3%)
 - Tier 2: Moderate Efficiency: 15029 items (23.6%)
 - Tier 3: Low Efficiency: 3899 items (6.1%)

 🎉 SUCCESS: 2026 Inference Complete!
 -> Output saved to 'Item_Zone_Efficiency_Predictions_2026.csv'
 -> Total Records Scored: 63816

--- Final Predictions Preview ---
  month storageSectionId   itemCode  CRITIC_Efficiency_Score  AHP_Efficiency_Score  Hybrid_Efficiency_Score
2026-01               AA 009331401A                    89.59                 93.93                    89.97
2026-01               AA 0128101431    